<a href="https://colab.research.google.com/github/VLCHS/FCNN_5_reactions/blob/main/FCNN_5_reactions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Get Started

In [ ]:
!pip install -q torch lightning comet_ml

In [ ]:
import os
import comet_ml
import torch
import random
import math
import numpy as np
import pandas as pd
import torch.nn as nn
import lightning as L
import matplotlib.pyplot as plt

from torchmetrics import MetricCollection, MeanAbsoluteError, MeanSquaredError
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
from torch.optim.lr_scheduler import ReduceLROnPlateau

from lightning.pytorch.callbacks import ModelCheckpoint, LearningRateMonitor
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from pytorch_lightning.loggers import CometLogger

from torch.utils.data import random_split
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.utils import resample

def set_random_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    random.seed(seed)
    L.seed_everything(seed)

set_random_seed(42)

In [ ]:
device = "gpu" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

In [ ]:
comet_ml.login()

In [ ]:
experiment_config = comet_ml.ExperimentConfig(name="n_pi_plus")
exp = comet_ml.start(project_name="val_test_mae", experiment_config=experiment_config)
run_name = exp.name

In [ ]:
params_dict = {
    'feature_scaler': StandardScaler(),
    'label_scaler': StandardScaler(),
    'batch_size': 256,
    'net_architecture': [5, 60, 80, 100, 120, 140, 240, 340, 440, 640, 2000, 1040, 640, 340, 240, 140, 100, 80, 60, 20, 1],
    'activation_function': nn.ReLU,
    # 'loss_func': 'RMSELoss()',
    'optim_func': torch.optim.Adam,
    'max_epochs': 200,
    'es_min_delta': 1e-05, #0.00001,
    'es_patience': 30,
    'lr': 0.001,
    'lr_factor':0.5,
    'lr_patience': 5,
    'lr_cooldown': 20,
}

# Подготовка данных

In [ ]:
from google.colab import files
uploaded = files.upload()

##$ \pi^{+} n$

In [ ]:
# pi_plus_n 00001
df_pi_plus_n = pd.read_csv('/content/clasdb_pi_plus_n.txt', delimiter='\t', header=None)
df_pi_plus_n.columns = ['Ebeam', 'W', 'Q2', 'cos_theta', 'phi', 'dsigma_dOmega', 'error', 'id']
df_pi_plus_n.loc[8314:65670, 'Ebeam'] = 5.754 # peculiarity of this dataset.
df_pi_plus_n['phi'] = df_pi_plus_n.phi.apply(lambda x: math.radians(x))
df_pi_plus_n['cos_phi'] = df_pi_plus_n['phi'].apply(lambda x: math.cos(x))
df_pi_plus_n['sin_phi'] = df_pi_plus_n['phi'].apply(lambda x: math.sin(x))

df_pi_plus_n = df_pi_plus_n.iloc[df_pi_plus_n[['Ebeam', 'W', 'Q2', 'cos_theta', 'phi']].drop_duplicates().index]
df_pi_plus_n = df_pi_plus_n[df_pi_plus_n.dsigma_dOmega <= df_pi_plus_n.dsigma_dOmega.quantile(0.96)]
df_pi_plus_n = df_pi_plus_n[df_pi_plus_n['error'] <= df_pi_plus_n["error"].quantile(0.96)]

df_pi_plus_n = df_pi_plus_n.drop('id', axis=1)
df_pi_plus_n = df_pi_plus_n.reset_index(drop=True)

# df_pi_plus_n['A'] = 1
# df_pi_plus_n['B'] = 0
# df_pi_plus_n['C'] = 0
# df_pi_plus_n['D'] = 0
# df_pi_plus_n['E'] = 1

df_pi_plus_n

##$ \pi^{0} p$

In [ ]:
# pi_0_p 00010
df_pi_0_p = pd.read_csv('/content/clasdb_pi_0_p.txt', delimiter='\t', header=None)
df_pi_0_p.columns = ['Ebeam', 'W', 'Q2', 'cos_theta', 'phi', 'dsigma_dOmega', 'error', 'id']
df_pi_0_p['phi'] = df_pi_0_p.phi.apply(lambda x: math.radians(x))
df_pi_0_p['cos_phi'] = df_pi_0_p['phi'].apply(lambda x: math.cos(x))
df_pi_0_p['sin_phi'] = df_pi_0_p['phi'].apply(lambda x: math.sin(x))
df_pi_0_p['Ebeam'] = df_pi_0_p['Ebeam'].round(decimals=2)
df_pi_0_p = df_pi_0_p.replace({"Ebeam": {2.45: 2.44, 1.65: 1.64}})

df_pi_0_p = df_pi_0_p.iloc[df_pi_0_p[['Ebeam', 'W', 'Q2', 'cos_theta', 'phi']].drop_duplicates().index]
df_pi_0_p = df_pi_0_p.drop(df_pi_0_p[df_pi_0_p['dsigma_dOmega'] == 0].index)
df_pi_0_p = df_pi_0_p.drop(df_pi_0_p[df_pi_0_p['dsigma_dOmega'] <= df_pi_0_p["error"]].index)
df_pi_0_p = df_pi_0_p[df_pi_0_p['dsigma_dOmega'] <= df_pi_0_p["dsigma_dOmega"].quantile(0.5)]
df_pi_0_p = df_pi_0_p[df_pi_0_p['error'] <= df_pi_0_p["error"].quantile(0.5)]

df_pi_0_p = df_pi_0_p.drop('id', axis=1)
df_pi_0_p = df_pi_0_p.reset_index(drop=True)


df_pi_0_p['A'] = 0
df_pi_0_p['B'] = 1
# df_pi_0_p['C'] = 0
# df_pi_0_p['D'] = 1
# df_pi_0_p['E'] = 0

df_pi_0_p

##$K^{+} Σ^{0}$

In [ ]:
# K_plus_Sigma_0 00100
df_K_plus_Sigma_0 = pd.read_csv('/content/K_plus_Sigma_0_Carman_5.499GeV.txt', delimiter='\t')
df_K_plus_Sigma_0.columns = ['Measurement', 'Title', 'Year', 'W', 'Q2', 'cos_theta', 'phi', 'dsigma_dOmega', 'error']
df_K_plus_Sigma_0 = df_K_plus_Sigma_0.drop(['Measurement', 'Title', 'Year'], axis=1)
df_K_plus_Sigma_0['phi'] = df_K_plus_Sigma_0.phi.apply(lambda x: math.radians(x))
df_K_plus_Sigma_0['cos_phi'] = df_K_plus_Sigma_0['phi'].apply(lambda x: math.cos(x))
df_K_plus_Sigma_0['sin_phi'] = df_K_plus_Sigma_0['phi'].apply(lambda x: math.sin(x))
df_K_plus_Sigma_0['Ebeam'] = 5.499

df_K_plus_Sigma_0 = df_K_plus_Sigma_0.iloc[df_K_plus_Sigma_0[['Ebeam', 'W', 'Q2', 'cos_theta', 'phi']].drop_duplicates().index]
#df_K_plus_Sigma_0 = df_K_plus_Sigma_0.drop(df_K_plus_Sigma_0[df_K_plus_Sigma_0['dsigma_dOmega'] == 0].index)

df_K_plus_Sigma_0 = df_K_plus_Sigma_0.reset_index(drop=True)

df_K_plus_Sigma_0['A'] = 0
df_K_plus_Sigma_0['B'] = 0
df_K_plus_Sigma_0['C'] = 1
df_K_plus_Sigma_0['D'] = 0
df_K_plus_Sigma_0['E'] = 0

df_K_plus_Sigma_0

In [ ]:
proton_mass = 0.93827

In [ ]:
# K_plus_Sigma_0 00100
df_K_plus_Sigma_0_eps = pd.read_csv('/content/K_plus_Sigma_0_epsilon.txt', delimiter='\t')
df_K_plus_Sigma_0_eps.columns = ['Measurement', 'Title', 'Year', 'W', 'Q2', 'epsilon', 'cos_theta', 'phi', 'dsigma_dOmega', 'error']

df_K_plus_Sigma_0_eps['Nu'] = (df_K_plus_Sigma_0_eps['W'] ** 2 - proton_mass ** 2 + df_K_plus_Sigma_0_eps['Q2'] ** 2) / (2 * proton_mass)
df_K_plus_Sigma_0_eps['Ebeam'] = df_K_plus_Sigma_0_eps['Nu']/2 + ((1+df_K_plus_Sigma_0_eps['epsilon'])*(df_K_plus_Sigma_0_eps['Nu']**2+df_K_plus_Sigma_0_eps['Q2'])/(4*(1-df_K_plus_Sigma_0_eps['epsilon'])))**0.5
df_K_plus_Sigma_0_eps['Ebeam'] = df_K_plus_Sigma_0_eps['Ebeam'].round(decimals=2)

df_K_plus_Sigma_0_eps = df_K_plus_Sigma_0_eps.drop(['Measurement', 'Title', 'Year', 'Nu', 'epsilon'], axis=1)
df_K_plus_Sigma_0_eps['phi'] = df_K_plus_Sigma_0_eps.phi.apply(lambda x: math.radians(x))
df_K_plus_Sigma_0_eps['cos_phi'] = df_K_plus_Sigma_0_eps['phi'].apply(lambda x: math.cos(x))
df_K_plus_Sigma_0_eps['sin_phi'] = df_K_plus_Sigma_0_eps['phi'].apply(lambda x: math.sin(x))

df_K_plus_Sigma_0_eps = df_K_plus_Sigma_0_eps.iloc[df_K_plus_Sigma_0_eps[['Ebeam', 'W', 'Q2', 'cos_theta', 'phi']].drop_duplicates().index]
#df_K_plus_Sigma_0_eps = df_K_plus_Sigma_0_eps.drop(df_K_plus_Sigma_0_eps[df_K_plus_Sigma_0_eps['dsigma_dOmega'] == 0].index)

df_K_plus_Sigma_0_eps = df_K_plus_Sigma_0_eps.reset_index(drop=True)

df_K_plus_Sigma_0_eps['A'] = 0
df_K_plus_Sigma_0_eps['B'] = 0
df_K_plus_Sigma_0_eps['C'] = 1
df_K_plus_Sigma_0_eps['D'] = 0
df_K_plus_Sigma_0_eps['E'] = 0

df_K_plus_Sigma_0_eps

##$K^{+} Λ$

In [ ]:
# df_K_plus_lambda 01000
df_K_plus_lambda = pd.read_csv('/content/K_plus_Lambda_Carman_5.499GeV.txt', delimiter='\t')
df_K_plus_lambda.columns = ['Measurement', 'Title', 'Year', 'W', 'Q2', 'cos_theta', 'phi', 'dsigma_dOmega', 'error']
df_K_plus_lambda = df_K_plus_lambda.drop(['Measurement', 'Title', 'Year'], axis=1)
df_K_plus_lambda['phi'] = df_K_plus_lambda.phi.apply(lambda x: math.radians(x))
df_K_plus_lambda['cos_phi'] = df_K_plus_lambda['phi'].apply(lambda x: math.cos(x))
df_K_plus_lambda['sin_phi'] = df_K_plus_lambda['phi'].apply(lambda x: math.sin(x))
df_K_plus_lambda['Ebeam'] = 5.499

df_K_plus_lambda = df_K_plus_lambda.iloc[df_K_plus_lambda[['Ebeam', 'W', 'Q2', 'cos_theta', 'phi']].drop_duplicates().index]
#df_K_plus_lambda = df_K_plus_lambda.drop(df_K_plus_lambda[df_K_plus_lambda['dsigma_dOmega'] == 0].index)

df_K_plus_lambda = df_K_plus_lambda.reset_index(drop=True)

df_K_plus_lambda['A'] = 0
df_K_plus_lambda['B'] = 1
df_K_plus_lambda['C'] = 0
df_K_plus_lambda['D'] = 0
df_K_plus_lambda['E'] = 0

df_K_plus_lambda

In [ ]:
# K_plus_lambda_eps 01000
df_K_plus_lambda_eps = pd.read_csv('/content/K_plus_Lambda_epsilon.txt', delimiter='\t')
df_K_plus_lambda_eps.columns = ['Measurement', 'Title', 'Year', 'W', 'Q2', 'epsilon', 'cos_theta', 'phi', 'dsigma_dOmega', 'error']

df_K_plus_lambda_eps['Nu'] = (df_K_plus_lambda_eps['W'] ** 2 - proton_mass ** 2 + df_K_plus_lambda_eps['Q2'] ** 2) / (2 * proton_mass)
df_K_plus_lambda_eps['Ebeam'] = df_K_plus_lambda_eps['Nu']/2 + ((1+df_K_plus_lambda_eps['epsilon'])*(df_K_plus_lambda_eps['Nu']**2+df_K_plus_lambda_eps['Q2'])/(4*(1-df_K_plus_lambda_eps['epsilon'])))**0.5
df_K_plus_lambda_eps['Ebeam'] = df_K_plus_lambda_eps['Ebeam'].round(decimals=2)

df_K_plus_lambda_eps = df_K_plus_lambda_eps.drop(['Measurement', 'Title', 'Year', 'Nu', 'epsilon'], axis=1)
df_K_plus_lambda_eps['phi'] = df_K_plus_lambda_eps.phi.apply(lambda x: math.radians(x))
df_K_plus_lambda_eps['cos_phi'] = df_K_plus_lambda_eps['phi'].apply(lambda x: math.cos(x))
df_K_plus_lambda_eps['sin_phi'] = df_K_plus_lambda_eps['phi'].apply(lambda x: math.sin(x))

df_K_plus_lambda_eps = df_K_plus_lambda_eps.iloc[df_K_plus_lambda_eps[['Ebeam', 'W', 'Q2', 'cos_theta', 'phi']].drop_duplicates().index]
#df_K_plus_lambda_eps = df_K_plus_lambda_eps.drop(df_K_plus_lambda_eps[df_K_plus_lambda_eps['dsigma_dOmega'] == 0].index)

df_K_plus_lambda_eps = df_K_plus_lambda_eps.reset_index(drop=True)

df_K_plus_lambda_eps['A'] = 0
df_K_plus_lambda_eps['B'] = 1
df_K_plus_lambda_eps['C'] = 0
df_K_plus_lambda_eps['D'] = 0
df_K_plus_lambda_eps['E'] = 0

df_K_plus_lambda_eps

##$η p$

In [ ]:
# eta_p 10000
df_eta_p1 = pd.read_csv('/content/eta_p_6cols.txt', delimiter='\t')
df_eta_p1.columns = ['Measurement', 'Title', 'Year', 'W', 'Q2', 'cos_theta', 'phi', 'dsigma_dOmega', 'error']

df_eta_p2 = pd.read_csv('/content/eta_p_7cols.txt', delimiter='\t')
df_eta_p2.columns = ['Measurement', 'Title', 'Year', 'W', 'Q2', 'cos_theta', 'phi', 'dsigma_dOmega', 'error', 'Systematic error']
df_eta_p2 = df_eta_p2.drop(['Systematic error'], axis=1)

df_eta_p = pd.concat([df_eta_p1, df_eta_p2], ignore_index=True)

df_eta_p['Ebeam'] = df_eta_p['Measurement'].map({
    "Measurement E102M1": 1.515,
    "Measurement E102M2": 1.515,
    "Measurement E102M3": 1.515,
    "Measurement E102M4": 1.515,
    "Measurement E102M5": 1.515,
    "Measurement E102M6": 1.515,

    "Measurement E102M7": 2.567,
    "Measurement E102M8": 2.567,
    "Measurement E102M9": 2.567,
    "Measurement E102M10": 2.567,
    "Measurement E102M11": 2.567,
    "Measurement E102M12": 2.567,
    "Measurement E102M13": 2.567,
    "Measurement E102M14": 2.567,
    "Measurement E102M15": 2.567,
    "Measurement E102M16": 2.567,
    "Measurement E102M17": 2.567,
    "Measurement E102M18": 2.567,
    "Measurement E102M19": 2.567,
    "Measurement E102M20": 2.567,
    "Measurement E102M21": 2.567,
    "Measurement E102M22": 2.567,
    "Measurement E102M23": 2.567,
    "Measurement E102M24": 2.567,
    "Measurement E102M25": 2.567,
    "Measurement E102M26": 2.567,
    "Measurement E102M27": 2.567,
    "Measurement E102M28": 2.567,
    "Measurement E102M29": 2.567,
    "Measurement E102M30": 2.567,
    "Measurement E102M31": 2.567,
    "Measurement E102M32": 2.567,
    "Measurement E102M33": 2.567,
    "Measurement E102M34": 2.567,
    "Measurement E102M35": 2.567,

    "Measurement E102M36": 4.056,
    "Measurement E102M37": 4.056,
    "Measurement E102M38": 4.056,
    "Measurement E102M39": 4.056,
    "Measurement E102M40": 4.056,
    "Measurement E102M41": 4.056,
    "Measurement E102M42": 4.056,
    "Measurement E102M43": 4.056,
    "Measurement E102M44": 4.056,
    "Measurement E102M45": 4.056,
    "Measurement E102M46": 4.056,
    "Measurement E102M47": 4.056,
    "Measurement E102M48": 4.056,
    "Measurement E102M49": 4.056,
    "Measurement E102M50": 4.056,
    "Measurement E102M51": 4.056,
    "Measurement E102M52": 4.056,
    "Measurement E102M53": 4.056,
    "Measurement E102M54": 4.056,
    "Measurement E102M55": 4.056,
    "Measurement E102M56": 4.056,
    "Measurement E102M57": 4.056,
    "Measurement E102M58": 4.056,
    "Measurement E102M59": 4.056,
    "Measurement E102M60": 4.056,
    "Measurement E102M61": 4.056,
    "Measurement E102M62": 4.056,
    "Measurement E102M63": 4.056,
    "Measurement E102M64": 4.056,
    "Measurement E102M65": 4.056,
    "Measurement E102M66": 4.056,
    "Measurement E102M67": 4.056,
    "Measurement E102M68": 4.056,
    "Measurement E102M69": 4.056,
    "Measurement E102M70": 4.056,
    "Measurement E102M71": 4.056,
    "Measurement E102M72": 4.056,
    "Measurement E102M73": 4.056,
    "Measurement E102M74": 4.056,
    "Measurement E102M75": 4.056,
    "Measurement E102M76": 4.056,
    "Measurement E102M77": 4.056,
    "Measurement E102M78": 4.056,
    "Measurement E102M79": 4.056,
    "Measurement E102M80": 4.056,
    "Measurement E102M81": 4.056,
    "Measurement E102M82": 4.056,
    "Measurement E102M83": 4.056,
})

df_eta_p

In [ ]:
df_eta_p = df_eta_p.drop(['Measurement', 'Title', 'Year'], axis=1)

df_eta_p['phi'] = df_eta_p.phi.apply(lambda x: math.radians(x))
df_eta_p['cos_phi'] = df_eta_p['phi'].apply(lambda x: math.cos(x))
df_eta_p['sin_phi'] = df_eta_p['phi'].apply(lambda x: math.sin(x))

df_eta_p = df_eta_p.iloc[df_eta_p[['Ebeam', 'W', 'Q2', 'cos_theta', 'phi']].drop_duplicates().index]
#df_eta_p = df_eta_p.drop(df_eta_p[df_eta_p['dsigma_dOmega'] == 0].index)

df_eta_p = df_eta_p.reset_index(drop=True)

df_eta_p['A'] = 1
df_eta_p['B'] = 0
df_eta_p['C'] = 0
df_eta_p['D'] = 0
df_eta_p['E'] = 0

df_eta_p

## EDA

In [ ]:
reactions_dict = {
    "1": "\u03B3 p \u27F6 \u03C0\u207A n",
    "2": "\u03B3 p \u27F6 \u03C0\u2070 p",
    "3": "\u03B3 p \u27F6 K\u207A \u03A3\u2070",
    "4": "\u03B3 p \u27F6 K\u207A \u039B",
    "5": "\u03B3 p \u27F6 \u03B7 p"
}

for key, value in reactions_dict.items():
  print("{0}: {1}".format(key,value))

n = input("Введите номер реакции: ")
reaction = reactions_dict.get(n)
print(f"Выбранная реакция: {reaction}")

In [ ]:
df_dict = {
    "1": df_pi_plus_n,
    "2": df_pi_0_p,
    "3": pd.concat([df_K_plus_Sigma_0, df_K_plus_Sigma_0_eps], ignore_index=True),
    "4": pd.concat([df_K_plus_lambda, df_K_plus_lambda_eps], ignore_index=True),
    "5": df_eta_p
}

df_eda = df_dict.get(n)
energies = df_eda.Ebeam.unique()

plt.figure(figsize=(12, 8))
for energy in (sorted(energies)):
    plt.scatter(df_eda[df_eda.Ebeam==energy].W, df_eda[df_eda.Ebeam==energy].Q2, label=f"E_e = {energy} GeV")

plt.legend()
plt.title(reaction)
plt.xlabel('W, GeV')
plt.ylabel('Q\u00B2, GeV\u00B2')

In [ ]:
import seaborn as sns

plt.figure(figsize=(8, 6))
sns.boxplot(x=df_eda['dsigma_dOmega'], width=4)
Q = df_eda["dsigma_dOmega"].quantile(0.99)
plt.suptitle("{}\n99% Quantile={}".format(reaction, f'{Q:.4f}'), fontsize=20)
plt.xlabel("d\u03C3_d\u03A9")
plt.show()

# Подготовка данных для нейронной сети

In [ ]:
df = df_pi_plus_n
#df = pd.concat([df_pi_plus_n, df_pi_0_p], ignore_index=True) #df_K_plus_Sigma_0, df_K_plus_Sigma_0_eps, df_K_plus_lambda, df_K_plus_lambda_eps, df_eta_p], ignore_index=True)
df

In [ ]:
df = df.drop(df[df['dsigma_dOmega'] == 0].index)
df = df.reset_index(drop=True)
df

In [ ]:
feature_columns = ["Ebeam", "W",	"Q2",	"cos_theta", "cos_phi"]
feature_data = df[feature_columns]
label_data = df['dsigma_dOmega']


#TRAIN TEST SPLIT
X_train, X_residual, y_train, y_residual = train_test_split(feature_data,
                                                            label_data,
                                                            test_size=0.66,   #0.66, #0.2,
                                                            random_state=42)

X_test, X_val, y_test, y_val = train_test_split(X_residual,
                                                y_residual,
                                                test_size=0.3,   #0.3, #0.5,
                                                random_state=42)
print(X_train, X_val, X_test)

scaler_feature = params_dict.get('feature_scaler')
X_train = X_train.reset_index(drop=True)
X_val = X_val.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)

#scale feature_data
columns_to_scale = list(X_train.columns)
X_train[columns_to_scale] = pd.DataFrame(scaler_feature.fit_transform(X_train[columns_to_scale]))
X_val[columns_to_scale] = pd.DataFrame(scaler_feature.transform(X_val[columns_to_scale]))
X_test[columns_to_scale] = pd.DataFrame(scaler_feature.transform(X_test[columns_to_scale]))

#scale label_data
scaler_target = params_dict.get('label_scaler')
y_train = pd.Series(scaler_target.fit_transform(y_train.to_frame())[:,0])
y_val = pd.Series(scaler_target.transform(y_val.to_frame())[:,0])
y_test = pd.Series(scaler_target.transform(y_test.to_frame())[:,0])

X_train = torch.tensor(X_train.values, dtype=torch.float32)
X_test = torch.tensor(X_test.values, dtype=torch.float32)
X_val = torch.tensor(X_val.values, dtype=torch.float32)

y_train = torch.tensor(y_train.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32)
y_val = torch.tensor(y_val.values, dtype=torch.float32)

train_data = TensorDataset(X_train, y_train)
val_data = TensorDataset(X_val, y_val)
test_data = TensorDataset(X_test, y_test)

train_dataloader = DataLoader(train_data, batch_size=params_dict.get('batch_size'), shuffle=True)
val_dataloader = DataLoader(val_data, batch_size=params_dict.get('batch_size'), shuffle=False)
test_dataloader = DataLoader(test_data, batch_size=params_dict.get('batch_size'), shuffle=False)

# Создание модели

In [ ]:
class NeuralNetwork(nn.Module):

    def __init__(self):
        super().__init__()

        self.net_architecture = params_dict.get('net_architecture')
        self.activation_function = params_dict.get('activation_function')

        self.network = nn.Sequential()
        for i in range(1,len(self.net_architecture)):
            self.network.append(nn.Linear(self.net_architecture[i-1], self.net_architecture[i]))
            if i!=len(self.net_architecture)-1:
                self.network.append(self.activation_function())

            else:
                pass

    def forward(self, x):
        return self.network(x)

In [ ]:
model = NeuralNetwork()

# Обучение

In [ ]:
class Pipeline(L.LightningModule):
    def __init__(self, model, params, scaler_target=None):
        super().__init__()

        self.model = model
        self.params = params
        self.criterion = torch.nn.MSELoss()
        self.optimizer = self.params.get('optim_func')
        self.scaler_target = scaler_target

        self.metrics = MetricCollection([
            MeanAbsoluteError(),
            MeanSquaredError(),
            #R2Score()
        ])

        self.save_hyperparameters(ignore=['model', 'scaler_target'])

        self.train_metrics = self.metrics.clone(postfix='/train')
        self.val_metrics = self.metrics.clone(postfix='/val')
        self.test_metrics = self.metrics.clone(postfix='/test')

    def configure_optimizers(self):
        optimizer = self.optimizer(self.parameters(), lr=self.params.get('lr'))

        lr_optim = ReduceLROnPlateau(optimizer = optimizer,
                                     mode = 'min',
                                     factor = self.params.get('lr_factor'),
                                     patience = self.params.get('lr_patience'),
                                     cooldown=self.params.get('lr_cooldown'),
                                     threshold=0.01
                                     )
        return {"optimizer": optimizer,
                "lr_scheduler": {
                    "scheduler": lr_optim,
                    "interval": "epoch",
                    "monitor": "val_loss",
                    "frequency": 2,
                    "name": 'lr_scheduler_monitoring'
                    },
                }

    def training_step(self, batch, batch_idx):
        x, y = batch
        out = self.model(x)
        loss = torch.sqrt(self.criterion(out.reshape(-1), y))


        if self.scaler_target is not None:

            out_real = out.detach().cpu().numpy().reshape(-1, 1)
            y_real = y.detach().cpu().numpy().reshape(-1, 1)
            out_real = self.scaler_target.inverse_transform(out_real)
            y_real = self.scaler_target.inverse_transform(y_real)
            out_real = torch.from_numpy(out_real).flatten().to(self.device)
            y_real = torch.from_numpy(y_real).flatten().to(self.device)

            self.train_metrics.update(out_real.reshape(-1), y_real)

        else:
            self.train_metrics.update(out.reshape(-1), y)

        self.log("train_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        return loss


    def validation_step(self, batch, batch_idx):
        x, y = batch
        out = self.model(x)
        loss = torch.sqrt(self.criterion(out.reshape(-1), y))

        if self.scaler_target is not None:

            out_real = out.detach().cpu().numpy().reshape(-1, 1)
            y_real = y.detach().cpu().numpy().reshape(-1, 1)
            out_real = self.scaler_target.inverse_transform(out_real)
            y_real = self.scaler_target.inverse_transform(y_real)
            out_real = torch.from_numpy(out_real).flatten().to(self.device)
            y_real = torch.from_numpy(y_real).flatten().to(self.device)

            self.val_metrics.update(out_real.reshape(-1), y_real)

        else:
            self.val_metrics.update(out.reshape(-1), y)

        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)


    def on_validation_epoch_end(self):
        self.log_dict(self.val_metrics.compute(), prog_bar=True)
        self.val_metrics.reset()

    def on_train_epoch_end(self):
        self.log_dict(self.train_metrics.compute(), prog_bar=True)
        self.train_metrics.reset()

    def test_step(self, batch, batch_idx):
        x, y = batch
        out = self.model(x)

        if self.scaler_target is not None:

            out_real = out.detach().cpu().numpy().reshape(-1, 1)
            y_real = y.detach().cpu().numpy().reshape(-1, 1)
            out_real = self.scaler_target.inverse_transform(out_real)
            y_real = self.scaler_target.inverse_transform(y_real)
            out_real = torch.from_numpy(out_real).flatten().to(self.device)
            y_real = torch.from_numpy(y_real).flatten().to(self.device)

            self.test_metrics.update(out_real.reshape(-1), y_real)
            self.test_labels.append(y_real.cpu())
            self.test_predictions.append(out_real.cpu())

        else:
            self.test_metrics.update(out.reshape(-1), y)
            self.test_labels.append(y.cpu())
            self.test_predictions.append(out.cpu())


    def on_test_start(self):
        self.test_labels = []
        self.test_predictions = []

    def on_test_epoch_end(self):
        self.log_dict(self.test_metrics.compute())
        self.test_metrics.reset()

        # all_labels = torch.cat(self.test_labels)
        # all_predictions = torch.cat(self.test_predictions)

        # self.results_df = pd.DataFrame({
        #     'true_label': all_labels.numpy().flatten(),
        #     'prediction': all_predictions.numpy().flatten()
        # })

    def predict_step(self, batch, batch_idx, dataloader_idx=0):
        x, y = batch
        out = self.model(x)
        return out

    def configure_callbacks(self):
        lr_monitor = LearningRateMonitor(logging_interval='epoch')

        early_stop_callback = EarlyStopping(monitor="val_loss", mode="min",
                                            min_delta=self.params.get('es_min_delta'),
                                            patience=self.params.get('es_patience'))

        return [lr_monitor, early_stop_callback]

In [ ]:
L.seed_everything(42)

model = NeuralNetwork()
pl_model = Pipeline(model, params=params_dict, scaler_target=scaler_target)
comet_logger = CometLogger()

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    dirpath=f"./checkpoints/run_{run_name}",
    filename="best-{epoch:02d}-{val_loss:.6f}"
)

trainer = L.Trainer(
    logger=comet_logger,
    max_epochs=params_dict.get("max_epochs"),
    accelerator=device,
    callbacks=[checkpoint_callback]
)

trainer.fit(
    model=pl_model,
    train_dataloaders=train_dataloader,
    val_dataloaders=val_dataloader
)

results WITH **scaler for targets**:

In [ ]:
best_ckpt_path = checkpoint_callback.best_model_path
best_ckpt_path

In [ ]:
# test_data
with torch.no_grad():
    predictions = trainer.predict(
        model=pl_model,
        dataloaders=test_dataloader,
        ckpt_path=best_ckpt_path,
        weights_only=False
    )

predictions = torch.cat(predictions, dim=0).cpu().numpy()
predictions = scaler_target.inverse_transform(predictions)

y_test = y_test.reshape(-1, 1)
y_test = scaler_target.inverse_transform(y_test)

mae_test = mean_absolute_error(y_test, predictions)
mse_test = mean_squared_error(y_test, predictions)

print("Results_on_test_data")
print("Mean Absolute Error: ", mae_test)
print("Mean Squared Error: ", mse_test)

In [ ]:
# val_data
with torch.no_grad():
    predictions = trainer.predict(
        model=pl_model,
        dataloaders=val_dataloader,
        ckpt_path=best_ckpt_path,
        weights_only=False
    )

predictions = torch.cat(predictions, dim=0).cpu().numpy()
predictions = scaler_target.inverse_transform(predictions)

y_val = y_val.reshape(-1, 1)
y_val= scaler_target.inverse_transform(y_val)

mae_val = mean_absolute_error(y_val, predictions)
mse_val = mean_squared_error(y_val, predictions)

print("Results_on_validation_data")
print("Mean Absolute Error: ", mae_val)
print("Mean Squared Error: ", mse_val)

In [ ]:
trainer.test(dataloaders=test_dataloader, ckpt_path='best', weights_only=False)

In [ ]:
trainer.test(dataloaders=val_dataloader, ckpt_path='best', weights_only=False)

In [ ]:
exp.log_parameters({"test_mae": mae_test, "test_mse": mse_test, "val_mae": mae_val, "val_mse": mse_val})
exp.end()

In [ ]:
# # without scaler for targets
# trainer.test(model=pl_model, dataloaders=test_dataloader)
# results_dataframe = pl_model.results_df
# print(results_dataframe)

# num_of_params = [5000, 4500, 181, 91, 101, 5.4, 6400, 40.8, 261, 1100]
# mae_list = [0.15968162, 0.14140316, 0.14223613, 0.17740805, 0.14791107, 0.16583956, 0.15021065, 0.16220232, 0.14119283, 0.14399641]

# plt.rcParams["figure.figsize"] = (10, 7)
# plt.scatter(num_of_params, mae_list)
# plt.grid()
# plt.title("MAE vs Number of Parameters")
# plt.xlabel("Number of Net Parameters, k")
# plt.ylabel("MAE")

# plt.show()

#The graph of 1 hidden layer net

In [ ]:
num_of_neurons = [1000, 10000, 100000, 1000000, 200000, 300000, 400000, 500000, 600000, 700000, 800000, 5000, 50000, 150000]
mae_list = [0.2407, 0.2574, 0.1807, 0.1953, 0.2040, 0.1736, 0.1774, 0.1734, 0.1860, 0.1908, 0.1775, 0.1985,  0.2295]
#mse_list = [0.2110, 0.2104, 0.1872, 0.2202, 0.1959, 0.1592, 0.1682, 0.1717, 0.1694, 0.2221, 0, 0.1853,  0.2027]

plt.rcParams["figure.figsize"] = (10, 7)
plt.scatter(num_of_neurons, mae_list)
plt.grid()
plt.title("MAE vs Number of Neurons, 1 hidden layer")
plt.xlabel("Number of Net Parameters, k")
plt.ylabel("MAE")

plt.show()

#The graph of 2 hidden layer net

In [ ]:
num_of_params = [4000000, 2300000, 1000000, 817000, 495000, 254000, 92400, 1700000, 2600000, 2900000, 1400000, 1200000, 3600000, 4400000]
mae_list = [0.1716, 0.1513, 0.1496, 0.1519, 0.1516, 0.1560, 0.1509, 0.1539, 0.1552,  0.1627, 0.1617, 0.1568, 0.1588, ]
#mse_list = [0.1754, 0.1591, 0.1552, 0.1449, 0.1524, 0.1481, 0.1458, 0.1484, 0.1557, 0.1847, 0.1681, 0.1598, 0.1611, ]

plt.rcParams["figure.figsize"] = (10, 7)
plt.scatter(num_of_params, mae_list)
plt.grid()
plt.title("MAE vs Number of Params, 2 hidden layer")
plt.xlabel("Number of Net Parameters, k")
plt.ylabel("MAE")

plt.show()

# The graph of 3 hidden layer net

In [ ]:
num_of_params = [504000, 725000, 986000, 1300000, 1600000, 2000000, 2400000, 2900000, 3400000, 3900000, 4500000, 5100000, 323000, 182000]
mae_list = [0.1401, 0.1466, 0.1489, 0.1466, 0.1471, 0.1439, 0.1529, 0.1445, 0.1433, 0.1446, 0.1446,  0.1513, 0.1476, 0.1525]
#mse_list = [0.1375, 0.1539, -, 0.1433, 0.1499, 0.1423, 0.1583, 0.1414, 0.1426, 0.1436, 0.1455, 0.1531, 0.1579, 0.1535]

plt.rcParams["figure.figsize"] = (10, 7)
plt.scatter(num_of_params, mae_list)
plt.grid()
plt.title("MAE vs Number of Params, 2 hidden layer")
plt.xlabel("Number of Net Parameters, k")
plt.ylabel("MAE")

plt.show()

## Ebeam = 1.515, mae = 0.6842
## Ebeam = 5.754, mae =  0.0783
## Ebeam = 5.499, nae =  0.0876

# Training

In [ ]:
n_epochs = 100

loss_fn = nn.L1Loss() #params_dict.get('loss_func')
optimizer = params_dict.get('optim_func')
optimizer = optimizer(params=model.parameters(), lr=params_dict.get('lr'))
lr_optim = ReduceLROnPlateau(optimizer = optimizer,
                                     mode = 'min',
                                     factor = params_dict.get('lr_factor'),
                                     patience = params_dict.get('lr_patience'),
                                     cooldown= params_dict.get('lr_cooldown'),
                                     threshold = 0.01)

In [ ]:
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    model.train()

    train_loss_mean = 0.0
    val_loss_mean = 0.0

    for X, y in train_dataloader:
        y_preds = model(X).squeeze()
        loss = loss_fn(y_preds, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss_mean += loss.item()

    train_loss_mean /= len(train_dataloader)
    train_losses.append(train_loss_mean)

    model.eval()

    with torch.inference_mode():
        for X, y in val_dataloader:
            val_preds = model(X).squeeze()
            val_loss = loss_fn(val_preds, y)

            val_loss_mean += val_loss.item()

        val_loss_mean /= len(val_dataloader)
        val_losses.append(val_loss_mean)

        if (epoch + 1) % 5 == 0:
            print(f'Epoch {epoch + 1} | Train Loss: {train_loss_mean:.2f} | Val Loss: {val_loss_mean:.2f}')

In [ ]:
predictions = DataLoader(test_data, batch_size=len(test_data), shuffle=False)

model.eval()
with torch.inference_mode():
    for X, y in predictions:
        y_preds = model(X)
        print(f'Mean Absolute Error = {mean_absolute_error(y, y_preds)}')